In [0]:
%run ../utils/__utils__

In [0]:
config = get_config()
catalog = config["catalog"]
tabella_origine = f"{catalog}.silver.veicoli_emissioni"
tabella         = f"{catalog}.gold.fct_emissioni_per_marca"
print(f"Leggo: {tabella_origine}")
print(f"Scrivo in: {tabella}")

In [0]:
def controlla_risultato():
    df = spark.table(tabella)
    print(f"Numero di marche: {df.count()}")
    print("Le 5 marche con emissioni medie più alte:")
    display(df.orderBy("avg_co2_g_km", ascending=False).limit(5))
    print("Le 5 marche con emissioni medie più basse:")
    display(df.orderBy("avg_co2_g_km").limit(5))

steps_list = [
    {"name": "crea_fct_emissioni_per_marca",
     "query": f"""
        CREATE OR REPLACE TABLE {tabella} AS
        SELECT
            make,
            COUNT(*)                                     AS vehicle_count,
            ROUND(AVG(co2_g_km), 1)                      AS avg_co2_g_km,
            MIN(co2_g_km)                                AS min_co2_g_km,
            MAX(co2_g_km)                                AS max_co2_g_km,
            ROUND(AVG(fuel_consumption_comb_l_100km), 1) AS avg_fuel_consumption_comb_l_100km,
            current_timestamp()                          AS data_caricamento
        FROM {tabella_origine}
        GROUP BY make
     """},
    {"name": "controlla_risultato", "query": controlla_risultato},
]

run_query_send_logs(steps_list)